# Step 4 — Build Feature Blocks @ baseline (t0)

## Objective
Build the *feature tables* (1 row per patient) using only information available **up to baseline (t0)**:
- **Vitals @ t0** (last pre-t0 measurement)
- **Respiratory (FVC) @ t0** (last pre-t0 measurement; best trial)
- **Final table**: baseline + vitals + FVC (merge by patient)
- **Coverage table** to document availability percentages per block

## Inputs (raw/interim)
- `01_data/interim/baseline_targets_slopes_v1.csv` *(or equivalent Step 3A file, 1 row per patient, includes `t0` and `subject_id`)*
- `01_data/raw/PROACT_VITALSIGNS.csv`
- `01_data/raw/PROACT_FVC.csv` *(and/or SVC, if applicable)*

## Outputs (interim/processed)
- `01_data/processed/features_vitals_t0.csv`
- `01_data/processed/features_fvc_t0.csv`
- `01_data/processed/features_baseline_v1.csv`
- `04_outputs/tables/step4_coverage_features.csv`

## Temporal principle (leakage-free)
All features are extracted with the rule:
> **use only records with delta ≤ t0**, choosing the **last measurement before baseline** (last pre-baseline).

<div style="padding:10px;border-left:6px solid #FF5F5D;">
<b>Note:</b> this notebook does not train models. It only prepares features and documents coverage to support methodological decisions in Step 5.
</div>


In [ ]:
import os
import numpy as np
import pandas as pd

# Project paths
RAW = os.path.join("..", "01_data", "raw")
INTERIM = os.path.join("..", "01_data", "interim")
PROCESSED = os.path.join("..", "01_data", "processed")
OUT_TABLES = os.path.join("..", "04_outputs", "tables")

os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(OUT_TABLES, exist_ok=True)

baseline_path = os.path.join(INTERIM, "baseline_table_ALSFRS_R.csv")
fvc_path = os.path.join(RAW, "PROACT_FVC.csv")
svc_path = os.path.join(RAW, "PROACT_SVC.csv")            # optional (not used in V1)
vitals_path = os.path.join(RAW, "PROACT_VITALSIGNS.csv")

base = pd.read_csv(baseline_path)
fvc = pd.read_csv(fvc_path)
vitals = pd.read_csv(vitals_path)

print("Baseline:", base.shape)
print("FVC:", fvc.shape)
print("VITALSIGNS:", vitals.shape)

base.head()


## 1) Utility functions and metric system normalisation

In this section we define functions to:
- convert units to the metric system (e.g., kg, cm)
- select the **last pre-t0 measurement** per patient (temporal rule)
- ensure column/name consistency throughout the pipeline

<b>Why this matters:</b> inconsistent units and incorrect temporal selection are two classic sources of noise and leakage. Here we ensure that each feature is "clinically interpretable" and temporally valid.


In [ ]:
def to_kg(weight, unit):
    if pd.isna(weight): 
        return np.nan
    u = str(unit).strip().lower() if not pd.isna(unit) else ""
    if u in ["kg", "kgs", "kilogram", "kilograms"]:
        return float(weight)
    if u in ["lb", "lbs", "pound", "pounds"]:
        return float(weight) * 0.45359237
    return float(weight)  # fallback (unknown unit)

def to_cm(height, unit):
    if pd.isna(height): 
        return np.nan
    u = str(unit).strip().lower() if not pd.isna(unit) else ""
    if u in ["cm", "centimeter", "centimeters"]:
        return float(height)
    if u in ["m", "meter", "meters"]:
        return float(height) * 100.0
    if u in ["in", "inch", "inches"]:
        return float(height) * 2.54
    if u in ["ft", "feet"]:
        return float(height) * 30.48
    return float(height)  # fallback

def last_prebaseline(df, delta_col, base_df, t0_col="t0_delta_days"):
    """Keep only records with delta <= t0 and return the last (closest to t0) per subject."""
    tmp = df.copy()
    tmp[delta_col] = pd.to_numeric(tmp[delta_col], errors="coerce")
    tmp = tmp.merge(base_df[["subject_id", t0_col]], on="subject_id", how="inner")
    tmp[t0_col] = pd.to_numeric(tmp[t0_col], errors="coerce")
    tmp = tmp.dropna(subset=[delta_col, t0_col])

    tmp = tmp[tmp[delta_col] <= tmp[t0_col]].copy()
    tmp = tmp.sort_values(["subject_id", delta_col], ascending=[True, True])
    # last record per subject
    return tmp.groupby("subject_id", as_index=False).tail(1).copy()


## 2) Build block: Vitals @ t0 (last pre-baseline measurement)

Process:
1) filter vital-sign records to delta ≤ t0 (pre-baseline)
2) select the last measurement per patient
3) convert to metric units (weight/height) and derive BMI (if applicable)
4) rename columns with `_t0` suffix to avoid ambiguity

<b>Output:</b> `features_vitals_t0.csv` (1 row per patient)


In [ ]:
v_last = last_prebaseline(vitals, "Vital_Signs_Delta", base)

# metric conversions
v_last["Weight_kg_t0"] = [to_kg(w,u) for w,u in zip(v_last.get("Weight"), v_last.get("Weight_Units"))]
v_last["Height_cm_t0"] = [to_cm(h,u) for h,u in zip(v_last.get("Height"), v_last.get("Height_Units"))]
v_last["BMI_t0"] = v_last["Weight_kg_t0"] / ((v_last["Height_cm_t0"]/100.0)**2)

# column selection (only those that exist in the CSV)
candidate_cols = [
    "subject_id", "Vital_Signs_Delta",
    "Weight_kg_t0", "Height_cm_t0", "BMI_t0",
    "Pulse", "Respiratory_Rate", "Temperature",
    "Blood_Pressure_Systolic", "Blood_Pressure_Diastolic",
    "Baseline_Standing_BP_Systolic", "Baseline_Standing_BP_Diastolic",
    "Baseline_Supine_BP_Systolic", "Baseline_Supine_BP_Diastolic"
]
cols = [c for c in candidate_cols if c in v_last.columns]
vitals_features = v_last[cols].copy().rename(columns={"Vital_Signs_Delta":"vitals_delta_days"})

out_vitals = os.path.join(PROCESSED, "features_vitals_t0.csv")
vitals_features.to_csv(out_vitals, index=False)

print("Saved:", out_vitals, "| shape:", vitals_features.shape)
vitals_features.head()


## 3) Build block: Respiratory function (FVC) @ t0

Process:
1) filter FVC records to delta ≤ t0
2) select the last measurement per patient
3) convert trials to numeric
4) consolidate into one value per patient (e.g., best trial = maximum across trials)
5) rename columns with `_t0` suffix

<b>Note:</b> FVC is clinically relevant, but typically has lower coverage than vitals; this block is therefore accompanied by a coverage table.


In [ ]:
f_last = last_prebaseline(fvc, "Forced_Vital_Capacity_Delta", base)

# convert trials to numeric
lit_cols = [c for c in ["Subject_Liters_Trial_1","Subject_Liters_Trial_2","Subject_Liters_Trial_3"] if c in f_last.columns]
pct_cols = [c for c in ["pct_of_Normal_Trial_1","pct_of_Normal_Trial_2","pct_of_Normal_Trial_3"] if c in f_last.columns]

for c in lit_cols + pct_cols:
    f_last[c] = pd.to_numeric(f_last[c], errors="coerce")

# best trial (maximum)
f_last["FVC_Liters_best_t0"] = f_last[lit_cols].max(axis=1) if lit_cols else np.nan
f_last["FVC_pctNormal_best_t0"] = f_last[pct_cols].max(axis=1) if pct_cols else np.nan

candidate_cols = [
    "subject_id", "Forced_Vital_Capacity_Delta",
    "FVC_Liters_best_t0", "FVC_pctNormal_best_t0",
    "subject_normal"
]
cols = [c for c in candidate_cols if c in f_last.columns]
fvc_features = f_last[cols].copy().rename(columns={"Forced_Vital_Capacity_Delta":"fvc_delta_days"})

out_fvc = os.path.join(PROCESSED, "features_fvc_t0.csv")
fvc_features.to_csv(out_fvc, index=False)

print("Saved:", out_fvc, "| shape:", fvc_features.shape)
fvc_features.head()


## 4) Build final feature table (baseline + vitals + FVC)

Here we perform a "horizontal" merge (1 row per patient):
- base (demographics/ALSFRS baseline and identification columns)
- +vitals_t0
- +fvc_t0

<b>Why keep blocks separate before merging:</b>
- allows documenting progress and decisions
- facilitates ablation study (baseline-only vs +vitals vs +FVC)
- makes it easier to identify coverage issues per block


In [ ]:
features_baseline_v1 = (
    base
    .merge(vitals_features, on="subject_id", how="left")
    .merge(fvc_features, on="subject_id", how="left")
)

out_final = os.path.join(PROCESSED, "features_baseline_v1.csv")
features_baseline_v1.to_csv(out_final, index=False)

print("Saved:", out_final, "| shape:", features_baseline_v1.shape)
features_baseline_v1.head()


## 5) Save coverage table (feature eligibility)

We create a coverage table per block:
- number of patients at baseline
- N and % with at least 1 available feature in Vitals
- N and % with at least 1 available feature in FVC
- N and % with data after the final merge

<b>Use in thesis:</b>
- justify why certain blocks are included/excluded from the model
- explain differences between 3m and 6m (effective sample size)
- support the "Threats to Validity" section (missingness/selection)


In [ ]:
N_base = base["subject_id"].nunique()

cov_tbl = pd.DataFrame({
    "feature_block": ["VITALSIGNS", "FVC", "FINAL (baseline+vitals+fvc)"],
    "subjects_with_any_data": [
        vitals_features["subject_id"].nunique(),
        fvc_features["subject_id"].nunique(),
        features_baseline_v1["subject_id"].nunique() 
    ]
})
cov_tbl["coverage_pct_of_baseline"] = cov_tbl["subjects_with_any_data"] / N_base * 100

out_cov = os.path.join(OUT_TABLES, "step4_featureblock_coverage.csv")
cov_tbl.to_csv(out_cov, index=False)

cov_tbl
